In [12]:
# Importing the libraries

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 

In [13]:
# Importing the training set
dataset_train = pd.read_csv('Google_Stock_Price_Train.csv')
training_set = dataset_train.iloc[:, 1:2].values

In [14]:
# Feature Scaling
from sklearn.preprocessing import MinMaxScaler
sc = MinMaxScaler(feature_range= (0,1))
training_set_scaled = sc.fit_transform(training_set)

In [15]:
# Create a datastructure with 60 timesteps and 1 output
x_train = []
y_train = []
for i in range(60,1256):
    x_train.append(training_set_scaled[i-60:i, 0])
    y_train.append(training_set_scaled[i, 0])
x_train,y_train = np.array(x_train), np.array(y_train)

# Reshoping
x_train = np.reshape(x_train,(x_train.shape[0],x_train.shape[1], 1))

In [16]:
# Importing the Keras libraries and packages
from keras.models import Sequential 
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Dropout 

In [17]:
# Initializing the RNN
regressor = Sequential()

# Adding the first LSTM layer and Dropout regularisation
regressor.add(LSTM(units = 50, return_sequences = True, input_shape = (x_train.shape[1], 1)))
regressor.add(Dropout(0.2))

# Adding a second LSTM layer and Dropout regularisation
regressor.add(LSTM(units = 50, return_sequences = True))
regressor.add(Dropout(0.2))

# Adding a third LSTM layer and Dropout regularisation
regressor.add(LSTM(units = 50, return_sequences = True))
regressor.add(Dropout(0.2))

# Adding the fourth LSTM layer and Dropout regularisation
regressor.add(LSTM(units = 50))
regressor.add(Dropout(0.2))

# Adding the output layer
regressor.add(Dense(units = 1))


e:\LANGCHAIN\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [18]:
# Compiling the RNN
regressor.compile(optimizer = 'adam' , loss = 'mean_squared_error')

# Feeting the RNN to the trainung set
regressor.fit(x_train , y_train , epochs= 100 , batch_size= 50) 

Epoch 1/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 81ms/step - loss: 0.0555
Epoch 2/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - loss: 0.0082
Epoch 3/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 68ms/step - loss: 0.0059
Epoch 4/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - loss: 0.0057
Epoch 5/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - loss: 0.0054
Epoch 6/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - loss: 0.0051
Epoch 7/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - loss: 0.0054
Epoch 8/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step - loss: 0.0045
Epoch 9/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 75ms/step - loss: 0.0049
Epoch 10/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 72ms/step - loss: 0.0048
Epoch 11/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - loss: 0.0046
Epoch 12/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - loss: 0.0061
Epoch 13/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - loss: 0.0049
Epoch 14/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step - loss: 0.0040
Epoch 15/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step - lo

In [19]:
# Part 3 -> Making the productions and visualizing the result

# Getting the real stuck price of 2017
dataset_test = pd.read_csv('Google_Stock_Price_Test.csv')
real_stock_price = dataset_test.iloc[:, 1:2].values 

In [22]:
# Getting the predicted stock price of 2017
dataset_total = pd.concat((dataset_train['Open'], dataset_test['Open']), axis = 0)
inputs = dataset_total[len(dataset_total) - len(dataset_test) - 60:].values
inputs = inputs.reshape(-1, 1)
inputs = sc.transform(inputs)

x_test = []
for i in range(60, 80):
    x_test.append(inputs[i-60:i, 0])  # Fixed: i-60 instead of -60

x_test = np.array(x_test)
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))

predicted_stock_price = regressor.predict(x_test)
predicted_stock_price = sc.inverse_transform(predicted_stock_price)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


In [ ]:
# Visualizing the result 
plt.plot(real_stock_price, color= 'red' , label = 'Reall Google Stock Price ')
plt.plot(predicted_stock_price , color = 'blue' , lebel = 'prediceted Google Stock Price ')
plt.title('Google')